# 08 — Web Platform | Data Export

Exports the Gold (and a few Silver) tables used by the web version of the platform (`web/index.html`) into one compact JSON file:
`/Volumes/rd_economic_intelligence/gold/web_exports/rd_platform_data.json`.

| Key | Source | Content |
|---|---|---|
| `fx` | `silver.bcrd_exchange_rates_daily` | Daily USD/DOP buy and sell since 1991 |
| `cpi` | `gold.inflation_metrics` | Index, MoM, YoY, YTD by series (headline, core, 12 groups, 4 regions) since 2018 |
| `col` | `gold.cost_of_living_metrics` | Family basket by quintile, national and region since 2018 |
| `fuel` | `silver.micm_fuel_prices_weekly` | Weekly prices of the five main fuels since 2018 |
| `con` | `gold.consumer_price_metrics` | Pro Consumidor product prices by month |
| `elec` | `gold.electricity_*` | Distributor indicators, BTS-1 bills and the current BTS-1 tariff |
| `agri` | `gold.agri_price_monthly`, `gold.agri_rubro_monthly` | Product-group series since 2017 and the last 25 months of product prices by channel and market |

Only aggregated, published statistics are exported (no source files). **Status:** Development.

In [0]:
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F

C = "rd_economic_intelligence"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {C}.gold.web_exports COMMENT 'Aggregated JSON exports for the web version of the platform'")
OUT = f"/Volumes/{C}/gold/web_exports/rd_platform_data.json"
num = lambda v, d=4: None if v is None else round(float(v), d)
ym = lambda d: d.strftime("%Y-%m")

# FX
fx = spark.table(f"{C}.silver.bcrd_exchange_rates_daily").orderBy("rate_date").select("rate_date", "buy_rate", "sell_rate").collect()
FX = {"d": [str(r[0]) for r in fx], "b": [num(r[1]) for r in fx], "s": [num(r[2]) for r in fx]}

# CPI
cpi = spark.table(f"{C}.gold.inflation_metrics").filter("reference_month >= '2018-01-01'").orderBy("series_type", "series_name", "reference_month").collect()
CPI = {}
for r in cpi:
    s = CPI.setdefault((r.series_type, r.series_name), {"t": r.series_type, "n": r.series_name, "m": [], "i": [], "mo": [], "yo": [], "yt": []})
    s["m"].append(ym(r.reference_month)); s["i"].append(num(r.index_value)); s["mo"].append(num(r.mom_inflation_pct, 3))
    s["yo"].append(num(r.yoy_inflation_pct, 3)); s["yt"].append(num(r.ytd_inflation_pct, 3))

# Cost of living
col = spark.table(f"{C}.gold.cost_of_living_metrics").filter("reference_month >= '2018-01-01'").orderBy("segment_type", "segment", "reference_month").collect()
COL = {}
for r in col:
    s = COL.setdefault((r.segment_type, r.segment), {"t": r.segment_type, "n": r.segment, "m": [], "c": [], "y": [], "r": []})
    s["m"].append(ym(r.reference_month)); s["c"].append(num(r.basket_cost_dop, 2)); s["y"].append(num(r.yoy_change_pct, 3)); s["r"].append(1 if r.is_retropolated else 0)

# Fuel
FUELS = [("GASOLINA PREMIUM", "Gasolina premium"), ("GASOLINA REGULAR", "Gasolina regular"), ("GASOIL OPTIMO", "Gasoil óptimo"), ("GASOIL REGULAR", "Gasoil regular"), ("GLP", "GLP")]
fu = (spark.table(f"{C}.silver.micm_fuel_prices_weekly").filter("period_start_date >= '2018-01-01'")
      .filter(F.col("fuel_name").isin([f[0] for f in FUELS])).select("period_start_date", "period_end_date", "fuel_name", "price_dop").collect())
weeks = sorted({(r[0], r[1]) for r in fu})
idx = {w[0]: i for i, w in enumerate(weeks)}
vals = {f[0]: [None] * len(weeks) for f in FUELS}
for r in fu:
    vals[r[2]][idx[r[0]]] = num(r[3], 2)
FUEL = {"w": [str(w[0]) for w in weeks], "e": [str(w[1]) for w in weeks], "f": [{"name": n, "values": vals[k]} for k, n in FUELS]}

# Pro Consumidor
con = spark.table(f"{C}.gold.consumer_price_metrics").orderBy("reference_month", "product", "presentation").collect()
CON = [[ym(r.reference_month), r.product, r.presentation, num(r.avg_price_dop, 2), num(r.min_price_dop, 2), num(r.max_price_dop, 2),
        r.cheapest_segment, r.most_expensive_segment, r.segments_observed, num(r.price_spread_pct, 2), num(r.mom_change_pct, 2), r.mom_segments_matched] for r in con]

# Electricity
em = spark.table(f"{C}.gold.electricity_metrics").orderBy("reference_month").collect()
months = sorted({ym(r.reference_month) for r in em})
D = {}
for r in em:
    s = D.setdefault(r.distributor, {k: [None] * len(months) for k in "pybslck"})
    i = months.index(ym(r.reference_month))
    for k, v in zip("pybslck", [r.selling_price_dop_kwh, r.selling_price_yoy_pct, r.purchase_price_cusd_kwh, r.selling_price_cusd_kwh,
                                 r.losses_rolling_pct, r.collection_rolling_pct, r.cri_rolling_pct]):
        s[k][i] = num(v)
bill = spark.table(f"{C}.gold.electricity_bts1_bill").orderBy("reference_month").collect()
bm = sorted({ym(r.reference_month) for r in bill}); B = {}
for r in bill:
    s = B.setdefault(f"{r.distributor}|{r.monthly_kwh}", {"a": [None] * len(bm), "r": [None] * len(bm)})
    i = bm.index(ym(r.reference_month)); s["a"][i] = num(r.applied_bill_dop, 2); s["r"][i] = num(r.reference_bill_dop, 2)
tar = spark.table(f"{C}.gold.electricity_tariff_monthly").filter(F.upper(F.regexp_replace("tariff", "[^A-Za-z0-9]", "")) == "BTS1")
tm = tar.agg(F.max("reference_month")).first()[0]
TAR = {"m": ym(tm), "rows": [[r.distributor, r.concept, num(r.applied_dop, 2), num(r.reference_dop, 2)]
                             for r in tar.filter(F.col("reference_month") == tm).orderBy("distributor", "concept").collect()]}
ELEC = {"m": months, "d": D, "bm": bm, "bill": B, "tar": TAR}

# Agriculture: product-group series (full history) + last 25 months of product prices
rub = spark.table(f"{C}.gold.agri_rubro_monthly").orderBy("channel", "rubro", "reference_month").collect()
RUB = {}
for r in rub:
    s = RUB.setdefault((r.channel, r.rubro), {"c": r.channel, "r": r.rubro, "m": [], "mom": [], "yoy": [], "n": [], "up": []})
    s["m"].append(ym(r.reference_month)); s["mom"].append(num(r.avg_mom_pct, 2)); s["yoy"].append(num(r.avg_yoy_pct, 2))
    s["n"].append(r.products_quoted); s["up"].append(num(r.share_up_pct, 1))
ap = spark.table(f"{C}.gold.agri_price_monthly")
last = ap.agg(F.max("reference_month")).first()[0]
first = F.add_months(F.lit(last), -24)
rows = ap.filter(F.col("reference_month") >= first).select("channel", "market", "rubro", "product", "unit", "reference_month",
                                                            "avg_price_dop", "yoy_pct", "volatility_12m_pct").collect()
am = sorted({ym(r.reference_month) for r in rows})
dicts = {k: sorted({getattr(r, k) for r in rows}) for k in ("market", "rubro", "product", "unit")}
pos = {k: {v: i for i, v in enumerate(vs)} for k, vs in dicts.items()}
S = {}
for r in rows:
    key = (r.channel, r.market, r.product, r.unit)
    s = S.setdefault(key, [0 if r.channel == "retail" else 1, pos["market"][r.market], pos["product"][r.product], pos["unit"][r.unit],
                           pos["rubro"][r.rubro], [None] * len(am), None, None])
    i = am.index(ym(r.reference_month)); s[5][i] = num(r.avg_price_dop, 2)
    if ym(r.reference_month) == am[-1]:
        s[6] = num(r.yoy_pct, 2); s[7] = num(r.volatility_12m_pct, 2)
AGRI = {"m": am, "markets": dicts["market"], "rubros": dicts["rubro"], "products": dicts["product"], "units": dicts["unit"],
        "s": list(S.values()), "rub": list(RUB.values())}

data = {"generated": datetime.now(timezone.utc).isoformat(timespec="seconds"), "fx": FX, "cpi": list(CPI.values()), "col": list(COL.values()),
        "fuel": FUEL, "con": CON, "elec": ELEC, "agri": AGRI}
text = json.dumps(data, ensure_ascii=False, separators=(",", ":"))
with open(OUT, "w", encoding="utf-8") as fh:
    fh.write(text)
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "path": OUT, "bytes": len(text.encode()), "fx_days": len(FX["d"]), "fx_range": [FX["d"][0], FX["d"][-1]],
    "cpi_last": CPI[("headline", "National CPI")]["m"][-1], "fuel_last": FUEL["w"][-1], "elec_last": months[-1],
    "agri": {"months": [am[0], am[-1]], "series": len(S), "rubro_series": len(RUB)}}))